# 02 - Binarização de Máscaras

Este notebook aplica o **método de binarização cientificamente validado** em todas as máscaras:

1. **Ground Truth** (JPG → PNG binário)
2. **Segmentações dos modelos** (PNG grayscale → PNG binário)

**Objetivo:** Garantir consistência na comparação, aplicando o mesmo pipeline de processamento (Gaussian Blur + Morphological Opening) em todas as máscaras.

## Imports

In [ ]:
import sys
from pathlib import Path

root_dir = Path.cwd()
if not (root_dir / 'src').exists() and (root_dir.parent / 'src').exists():
    root_dir = root_dir.parent

if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

import os
import numpy as np
from PIL import Image
from scipy.ndimage import gaussian_filter, binary_opening

from src.io.indice_loader import carregar_indice_excel
from src.io.path_utils import caminho_ground_truth
from src.config import (
    GROUND_OF_TRUTH_OUTPUT,
    MODELOS_PARA_AVALIACAO,
    SEGMENTED_RAW_DIR,
    SEGMENTED_BINARIZED_DIR,
)

## Leitura do índice

In [ ]:
indice_excel = carregar_indice_excel()

## Verificação de integridade dos PNGs raw

Valida todos os arquivos .png em `generated/segmentada/raw/` antes de processar. Remove arquivos corrompidos se encontrados.

In [ ]:
# Verifica integridade dos PNGs em generated/segmentada/raw/ e remove arquivos corrompidos
def verificar_e_limpar_pngs_corrompidos(diretorio_base):
    total_png = 0
    arquivos_integros = 0
    arquivos_removidos = 0
    falhas_remocao = 0

    for raiz, _, arquivos in os.walk(diretorio_base):
        for nome_arquivo in arquivos:
            if not nome_arquivo.lower().endswith('.png'):
                continue

            total_png += 1
            caminho_arquivo = os.path.join(raiz, nome_arquivo)

            try:
                with Image.open(caminho_arquivo) as img:
                    img.verify()

                with Image.open(caminho_arquivo) as img:
                    img.load()

                arquivos_integros += 1
            except Exception as e:
                print(f"Corrompido detectado: {caminho_arquivo}")
                print(f" - Erro: {e}")

                try:
                    os.remove(caminho_arquivo)
                    arquivos_removidos += 1
                    print(" - Arquivo removido com sucesso.")
                except OSError as erro_remocao:
                    falhas_remocao += 1
                    print(f" - Falha ao remover arquivo: {erro_remocao}")

    print('\nVerificacao de integridade concluida.')
    print(f" - Total de PNGs verificados: {total_png}")
    print(f" - Arquivos integros: {arquivos_integros}")
    print(f" - Arquivos removidos por corrupcao: {arquivos_removidos}")
    print(f" - Falhas ao remover: {falhas_remocao}")

if os.path.isdir(SEGMENTED_RAW_DIR):
    verificar_e_limpar_pngs_corrompidos(SEGMENTED_RAW_DIR)
else:
    print(f"AVISO: Diretório raw não encontrado: {SEGMENTED_RAW_DIR}")
    print("Execute o notebook 01 primeiro para gerar as máscaras raw.")

## Conversão do Ground Truth para PNG Binarizado

Esta seção implementa a conversão das máscaras Ground Truth (JPG → PNG binário) utilizando o **método cientificamente validado** através de análise comparativa.

### Problema: Artefatos de Compressão JPG

O formato JPG utiliza compressão com perdas que gera **artefatos nas bordas** (pixels cinzas intermediários onde deveria haver transição abrupta preto/branco). Ao aplicar um limiar simples (threshold), esses artefatos causam **bordas serrilhadas** que aumentam artificialmente o **perímetro** calculado, prejudicando análises morfométricas.

### Método Selecionado: Blur Gaussiano + Morphological Opening

Baseado em análise científica comparativa (notebook `selecao_metodo_binarizacao.ipynb`), o método **Gaussian Blur + Morphological Opening** foi selecionado como padrão de produção com **pontuação 96.56/100**.

**Critérios de avaliação** (387 imagens):
- **Consistência** (40%): Coeficiente de variação entre máscaras - mede estabilidade do método
- **Circularidade** (30%): Proximidade à forma circular ideal (4πA/P²) - detecta distorções
- **Suavidade de bordas** (30%): Desvio padrão da curvatura - mede qualidade das bordas

**Por que este método venceu:**
- Melhor equilíbrio entre remoção de ruído e preservação de detalhes anatômicos
- Bordas mais suaves (100/100 em suavidade)
- Forma mais consistente (100/100 em circularidade)
- Alta estabilidade entre imagens (91.40/100 em consistência)

**Parâmetros utilizados:**
- `sigma=1.0`: Intensidade do blur gaussiano (suaviza artefatos JPG)
- `threshold=127`: Limiar de binarização (ponto médio da escala 0-255)
- `kernel_size=3`: Tamanho do elemento estruturante para opening (remove pixels isolados)

**Comparação com métodos rejeitados:**
- Limiar simples (0.00/100): Bordas muito serrilhadas, alta variabilidade
- Blur gaussiano (74.04/100): Bom mas sem remoção de ruído isolado
- Filtro bilateral (74.81/100): Preserva bordas mas mais lento e menos consistente

### Implementação - Ground Truth

**Pipeline de processamento:**
1. Carrega máscara JPG original
2. Converte para escala de cinza (8-bit grayscale)
3. Aplica **Gaussian Blur** (sigma=1.0) para suavizar artefatos de compressão
4. Aplica **threshold** em 127 para binarização (0 ou 255)
5. Aplica **Morphological Opening** (erosão + dilatação) para remover pixels isolados
6. Salva resultado como PNG binário

**Output:** As máscaras binarizadas são salvas em `generated/ground-of-truth/*.png`

**Detalhes técnicos:**
- **Gaussian Filter** (`scipy.ndimage.gaussian_filter`): Remove ruído de alta frequência (artefatos JPG)
- **Binary Opening** (`scipy.ndimage.binary_opening`): Remove pequenos objetos e suaviza contornos
- **Elemento estruturante**: Matriz 3×3 de valores True (conectividade completa)

In [ ]:
# ===== PARÂMETROS =====
SIGMA = 1.0
LIMIAR = 127
KERNEL_SIZE = 3

# ===== CONVERSÃO =====
output_dir = GROUND_OF_TRUTH_OUTPUT
os.makedirs(output_dir, exist_ok=True)

print("Convertendo Ground Truth para PNG binarizado")
print(f"Método: Gaussian Blur + Morphological Opening")
print(f"Parâmetros: SIGMA={SIGMA}, LIMIAR={LIMIAR}, KERNEL_SIZE={KERNEL_SIZE}")
print(f"Total de imagens: {len(indice_excel)}\n")

processadas = 0
puladas = 0
erros = 0

for idx, linha in enumerate(indice_excel, 1):
    caminho_original = caminho_ground_truth(linha.nome_arquivo)
    caminho_saida = os.path.join(output_dir, f"{linha.nome_arquivo}.png")
    
    if not os.path.isfile(caminho_original):
        print(f"[{idx}/{len(indice_excel)}] ERRO: Máscara não encontrada: {caminho_original}")
        erros += 1
        continue
    
    if os.path.isfile(caminho_saida):
        puladas += 1
        continue
    
    try:
        with Image.open(caminho_original) as img:
            img_gray = img.convert('L')
            matriz = np.array(img_gray, dtype=np.float32)
            
            # 1. Aplica filtro gaussiano
            matriz_blur = gaussian_filter(matriz, sigma=SIGMA)
            
            # 2. Binariza
            matriz_binaria = (matriz_blur > LIMIAR).astype(bool)
            
            # 3. Morphological opening (remove ruído pequeno)
            estrutura = np.ones((KERNEL_SIZE, KERNEL_SIZE), dtype=bool)
            matriz_opening = binary_opening(matriz_binaria, structure=estrutura)
            
            # 4. Converte para uint8
            matriz_final = (matriz_opening * 255).astype(np.uint8)
            img_binarizada = Image.fromarray(matriz_final, mode='L')
            img_binarizada.save(caminho_saida, format='PNG')
        
        processadas += 1
        if processadas % 10 == 0 or processadas == len(indice_excel):
            print(f"[{idx}/{len(indice_excel)}] Processadas: {processadas} | Puladas: {puladas}")
    
    except Exception as e:
        print(f"[{idx}/{len(indice_excel)}] ERRO ao processar {linha.nome_arquivo}: {e}")
        erros += 1

print(f"\n{'='*60}")
print("Conversão CONCLUÍDA")
print(f"Processadas: {processadas} | Puladas: {puladas} | Erros: {erros}")
print(f"Salvo em: {output_dir}")
print(f"{'='*60}")

## Binarização das Máscaras dos Modelos

Aplica o **mesmo método** nas máscaras geradas pelos modelos, garantindo consistência científica.

### Pipeline de Processamento

1. Lê máscara raw (grayscale 0-255) de `generated/segmentada/raw/<modelo>/`
2. Aplica **Gaussian Filter** (sigma=1.0) - suaviza transições
3. Aplica **threshold** (127) - binarização
4. Aplica **Morphological Opening** (kernel 3×3) - remove ruído isolado
5. Salva em `generated/segmentada/binarized/<modelo>/`

### Parâmetros (mesmos do Ground Truth)

- `SIGMA = 1.0`
- `LIMIAR = 127`
- `KERNEL_SIZE = 3`

In [ ]:
# ===== PARÂMETROS (mesmos do GT) =====
SIGMA = 1.0
LIMIAR = 127
KERNEL_SIZE = 3

print("Binarizando máscaras dos modelos")
print(f"Método: Gaussian Blur + Morphological Opening")
print(f"Parâmetros: SIGMA={SIGMA}, LIMIAR={LIMIAR}, KERNEL_SIZE={KERNEL_SIZE}\n")

for modelo in MODELOS_PARA_AVALIACAO.keys():
    print(f"\n{'='*60}")
    print(f"Modelo: {modelo}")
    print(f"{'='*60}")
    
    input_dir = os.path.join(SEGMENTED_RAW_DIR, modelo)
    output_dir = os.path.join(SEGMENTED_BINARIZED_DIR, modelo)
    os.makedirs(output_dir, exist_ok=True)
    
    # Verifica se diretório raw existe
    if not os.path.isdir(input_dir):
        print(f"AVISO: Diretório raw não encontrado: {input_dir}")
        print(f"Execute o notebook 01 primeiro para gerar as máscaras raw.")
        continue
    
    processadas = 0
    puladas = 0
    erros = 0
    
    for idx, linha in enumerate(indice_excel, 1):
        caminho_entrada = os.path.join(input_dir, f"{linha.nome_arquivo}.png")
        caminho_saida = os.path.join(output_dir, f"{linha.nome_arquivo}.png")
        
        if not os.path.isfile(caminho_entrada):
            print(f"[{idx}/{len(indice_excel)}] ERRO: Máscara raw não encontrada: {caminho_entrada}")
            erros += 1
            continue
        
        if os.path.isfile(caminho_saida):
            puladas += 1
            continue
        
        try:
            with Image.open(caminho_entrada) as img:
                # Converte para grayscale se necessário
                if img.mode != 'L':
                    img = img.convert('L')
                
                matriz = np.array(img, dtype=np.float32)
                
                # 1. Aplica filtro gaussiano
                matriz_blur = gaussian_filter(matriz, sigma=SIGMA)
                
                # 2. Binariza
                matriz_binaria = (matriz_blur > LIMIAR).astype(bool)
                
                # 3. Morphological opening
                estrutura = np.ones((KERNEL_SIZE, KERNEL_SIZE), dtype=bool)
                matriz_opening = binary_opening(matriz_binaria, structure=estrutura)
                
                # 4. Converte para uint8
                matriz_final = (matriz_opening * 255).astype(np.uint8)
                img_binarizada = Image.fromarray(matriz_final, mode='L')
                img_binarizada.save(caminho_saida, format='PNG')
            
            processadas += 1
            if processadas % 10 == 0 or processadas == len(indice_excel):
                print(f"[{modelo}] Processadas: {processadas} | Puladas: {puladas}")
        
        except Exception as e:
            print(f"[{idx}/{len(indice_excel)}] ERRO ao processar {linha.nome_arquivo}: {e}")
            erros += 1
    
    print(f"\n{modelo} - Processadas: {processadas} | Puladas: {puladas} | Erros: {erros}")

print(f"\n{'='*60}")
print("Binarização de todos os modelos CONCLUÍDA")
print(f"Máscaras salvas em: {SEGMENTED_BINARIZED_DIR}")
print(f"{'='*60}")